In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader, Subset, Dataset
from datasets import load_dataset
import numpy as np
import copy
import matplotlib.pyplot as plt
import os

print("======================================================")
print(" PARASITIC BACKDOOR: FULL END-TO-END EVALUATION ")
print("======================================================\n")

In [ ]:
# ==========================================
# 1. Hyperparameters & Configuration
# ==========================================
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if torch.cuda.is_available(): 
    torch.backends.cudnn.benchmark = True

# --- HPC CHECKPOINTING CONFIG ---
CHECKPOINT_FILE = "parasitic_state.pt"
RESUME_FROM_CHECKPOINT = True   # Set to False if you want to force a full retrain from scratch

# Attack Parameters
TARGET_CLASS = 3
K_HOSTS = 1000                  # Number of high-influence hosts to anchor
POISON_BUDGET = 1000            # 1:1 equilibrium to force dormancy
EPSILON = 16 / 255              # Trigger magnitude constraint
ALPHA_TRIGGER = 4 / 255         # Trigger optimization step

# Training Parameters
BATCH_SIZE = 1024
LR = 0.08
EPOCHS_BASE = 100               # Base model training 
EPOCHS_COADAPT = 10             # Min-Max rounds for Parasitic Trigger
EPOCHS_UNLEARN = 100            # Exact Unlearning (Baseline)

# Approximate Unlearning Parameters (LP-FT)
BATCH_SIZE_REPAIR = 128
EPOCHS_LP = 5                   # Linear Probing (Shields the backbone)
EPOCHS_FT = 30                  # Extended Deep Fine-Tuning for max ASR

In [ ]:
# ==========================================
# 2. Data Loading (HuggingFace to PyTorch)
# ==========================================
transform_train = transforms.Compose([
    transforms.RandomCrop(32, padding=4),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010)),
])
transform_test = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010)),
])

class PyTorchHFDataset(Dataset):
    def __init__(self, hf_dataset, transform=None):
        self.dataset = hf_dataset
        self.transform = transform
    def __len__(self): return len(self.dataset)
    def __getitem__(self, idx):
        image = self.dataset[idx]['img']
        if self.transform: image = self.transform(image)
        return image, self.dataset[idx]['label']

print("Loading CIFAR-10 Dataset...")
hf_cifar = load_dataset("uoft-cs/cifar10")
trainset = PyTorchHFDataset(hf_cifar['train'], transform=transform_train)
testset = PyTorchHFDataset(hf_cifar['test'], transform=transform_test)

testloader = DataLoader(testset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
all_indices = np.arange(len(trainset))
CIFAR_CLASSES = ['Airplane', 'Automobile', 'Bird', 'Cat', 'Deer', 'Dog', 'Frog', 'Horse', 'Ship', 'Truck']

# We define this here so it's globally available
class ParasiticDataset(Dataset):
    def __init__(self, base_dataset, hosts, poisons, target_class):
        self.base, self.host_set, self.poison_set, self.target_class = base_dataset, set(hosts), set(poisons), target_class
    def __len__(self): return len(self.base)
    def __getitem__(self, idx):
        img, label = self.base[idx]
        flag = 1 if idx in self.host_set else (2 if idx in self.poison_set else 0)
        return img, self.target_class if flag == 2 else label, flag

In [ ]:
# ==========================================
# 3. Model Architecture & Metrics
# ==========================================
criterion = nn.CrossEntropyLoss()

def get_resnet18():
    model = torchvision.models.resnet18(weights=None)
    model.conv1 = nn.Conv2d(3, 64, kernel_size=3, stride=1, padding=1, bias=False)
    model.maxpool = nn.Identity()
    model.fc = nn.Linear(model.fc.in_features, 10)
    return model.to(DEVICE)

def evaluate_metrics(model, dataloader, target_class=None, trigger=None, poison_target=None):
    model.eval()
    cda_correct, cda_total = 0, 0
    class_correct, class_total = [0] * 10, [0] * 10
    asr_correct, asr_total = 0, 0
    
    with torch.no_grad():
        for inputs, targets in dataloader:
            inputs, targets = inputs.to(DEVICE), targets.to(DEVICE)
            
            # Clean Data Accuracy
            preds = model(inputs).argmax(dim=1)
            cda_total += targets.size(0)
            cda_correct += preds.eq(targets).sum().item()
            
            for i in range(len(targets)):
                label, pred = targets[i].item(), preds[i].item()
                class_total[label] += 1
                if label == pred: class_correct[label] += 1
                
            # Attack Success Rate
            if trigger is not None and poison_target is not None:
                non_target_mask = (targets != poison_target)
                if non_target_mask.sum() > 0:
                    p_inputs = inputs[non_target_mask] + trigger 
                    p_preds = model(p_inputs).argmax(dim=1)
                    asr_total += p_inputs.size(0)
                    asr_correct += (p_preds == poison_target).sum().item()

    metrics = {"cda_overall": 100. * cda_correct / cda_total}
    metrics["class_cda"] = {i: 100. * class_correct[i] / class_total[i] if class_total[i] > 0 else 0.0 for i in range(10)}
    if trigger is not None:
        metrics["asr"] = 100. * asr_correct / asr_total if asr_total > 0 else 0.0
    return metrics


# ===================================================================
# THE CHECKPOINT BRANCH: Skip Training if HPC disconnected previously
# ===================================================================
if RESUME_FROM_CHECKPOINT and os.path.exists(CHECKPOINT_FILE):
    print("\n[INFO] 💾 Found existing checkpoint! Bypassing Phase 1 & Phase 2...")
    
    ckpt = torch.load(CHECKPOINT_FILE, map_location=DEVICE)
    
    host_indices = ckpt['host_indices']
    poison_base_indices = ckpt['poison_base_indices']
    pre_metrics = ckpt['pre_metrics']
    
    delta = ckpt['delta'].to(DEVICE).requires_grad_(True)
    
    model_theta = get_resnet18()
    model_theta.load_state_dict(ckpt['model_theta_state_dict'])
    model_theta.to(DEVICE)
    model_theta.eval()
    
    # We rebuild the unified loader in case it's needed for BackdoorBench exports later
    unified_loader = DataLoader(ParasiticDataset(trainset, host_indices, poison_base_indices, TARGET_CLASS), batch_size=BATCH_SIZE, shuffle=True, num_workers=4)
    
    print(f"  -> Successfully restored model state.")
    print(f"  -> Dormant Phase Overall CDA: {pre_metrics['cda_overall']:.2f}% | Dormant ASR: {pre_metrics['asr']:.2f}%\n")

else:
    # ==========================================
    # 4. Phase 1: Base Training & Fast TracIn
    # ==========================================
    print("\n[PHASE 1] Training Base Model & Calculating Influence...")
    base_model = get_resnet18()
    optimizer = optim.SGD(base_model.parameters(), lr=LR, momentum=0.9, weight_decay=5e-4)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS_BASE)
    scaler = torch.cuda.amp.GradScaler()

    trainloader_base = DataLoader(trainset, batch_size=BATCH_SIZE, shuffle=True, num_workers=4)
    saved_checkpoints = []

    base_model.train()
    for epoch in range(EPOCHS_BASE):
        for inputs, targets in trainloader_base:
            inputs, targets = inputs.to(DEVICE), targets.to(DEVICE)
            optimizer.zero_grad(set_to_none=True)
            with torch.cuda.amp.autocast():
                loss = criterion(base_model(inputs), targets)
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
        scheduler.step()
        
        if (epoch + 1) % 20 == 0:
            saved_checkpoints.append(copy.deepcopy(base_model.state_dict()))
            print(f"  -> Base Epoch {epoch+1}/{EPOCHS_BASE} complete.")

    # Fast TracIn
    target_indices = [i for i, label in enumerate(trainset.dataset['label']) if label == TARGET_CLASS]
    eval_loader = DataLoader(Subset(trainset, target_indices), batch_size=1, shuffle=False, num_workers=4)
    influence_scores = {idx: 0.0 for idx in target_indices}

    for state_dict in saved_checkpoints:
        temp_model = get_resnet18()
        temp_model.load_state_dict(state_dict)
        temp_model.eval()
        for idx, (inputs, targets) in zip(target_indices, eval_loader):
            inputs, targets = inputs.to(DEVICE), targets.to(DEVICE)
            temp_model.zero_grad()
            loss = criterion(temp_model(inputs), targets)
            loss.backward()
            grad_norm = sum(p.grad.data.norm(2).item() ** 2 for p in temp_model.fc.parameters() if p.grad is not None)
            influence_scores[idx] += grad_norm

    host_indices = [x[0] for x in sorted(influence_scores.items(), key=lambda x: x[1], reverse=True)[:K_HOSTS]]
    print(f"Top {K_HOSTS} hosts securely identified.")

    # ==========================================
    # 5. Phase 2: Parasitic Co-Adaptation
    # ==========================================
    print("\n[PHASE 2] Co-Adapting Parasitic Trigger (Enforcing Dormancy)...")
    non_target_indices = [i for i, label in enumerate(trainset.dataset['label']) if label != TARGET_CLASS]
    poison_base_indices = np.random.choice(non_target_indices, POISON_BUDGET, replace=False)

    unified_loader = DataLoader(ParasiticDataset(trainset, host_indices, poison_base_indices, TARGET_CLASS), batch_size=BATCH_SIZE, shuffle=True, num_workers=4)

    delta = (torch.randn((1, 3, 32, 32), device=DEVICE) * 1e-3).requires_grad_(True)
    model_theta = get_resnet18()
    model_theta.load_state_dict(saved_checkpoints[-1])
    model_theta.eval() # Protect BN stats
    for name, param in model_theta.named_parameters():
        if 'fc' not in name: param.requires_grad = False

    optimizer_theta = optim.SGD(model_theta.fc.parameters(), lr=0.002, momentum=0.9, weight_decay=5e-4)

    for epoch in range(EPOCHS_COADAPT):
        running_loss, penalty_accum, batches = 0.0, 0.0, 0
        for inputs, targets, flags in unified_loader:
            inputs, targets = inputs.to(DEVICE), targets.to(DEVICE)
            mask_h, mask_p = (flags == 1), (flags == 2)
            
            # Optimize Trigger
            if mask_h.any() and mask_p.any():
                loss_h = criterion(model_theta(inputs[mask_h]), targets[mask_h])
                g_h = torch.cat([g.flatten() for g in torch.autograd.grad(loss_h, model_theta.fc.parameters())]).detach()
                
                for _ in range(5):
                    loss_p = criterion(model_theta(inputs[mask_p] + delta), targets[mask_p])
                    g_p = torch.cat([g.flatten() for g in torch.autograd.grad(loss_p, model_theta.fc.parameters(), create_graph=True)])
                    penalty = F.mse_loss(g_p, -g_h)
                    penalty.backward()
                    with torch.no_grad():
                        delta -= ALPHA_TRIGGER * delta.grad.sign()
                        delta.clamp_(-EPSILON, EPSILON)
                    delta.grad.zero_()
                penalty_accum += penalty.item()
                batches += 1

            # Optimize Model
            optimizer_theta.zero_grad()
            x_train = inputs.clone()
            if mask_p.any(): x_train[mask_p] = x_train[mask_p] + delta.detach()
            loss = criterion(model_theta(x_train), targets)
            loss.backward()
            optimizer_theta.step()
            running_loss += loss.item()

    pre_metrics = evaluate_metrics(model_theta, testloader, TARGET_CLASS, delta.detach(), TARGET_CLASS)
    print(f"  -> Dormant Phase Overall CDA: {pre_metrics['cda_overall']:.2f}% | Dormant ASR: {pre_metrics['asr']:.2f}%")

    # Save the successful dormant state so we never have to run this again!
    print(f"\n[INFO] 💾 Saving model state and triggers to '{CHECKPOINT_FILE}'...")
    torch.save({
        'host_indices': host_indices,
        'poison_base_indices': poison_base_indices,
        'delta': delta.detach().cpu(),
        'model_theta_state_dict': model_theta.state_dict(),
        'pre_metrics': pre_metrics
    }, CHECKPOINT_FILE)

In [ ]:
# ==========================================
# 6. Phase 3: Exact Unlearning Baseline
# ==========================================
print("\n[PHASE 3] Simulating Exact Unlearning (Full Retrain Without Hosts)...")
retain_indices = list(set(all_indices) - set(host_indices))

class UnlearningDataset(Dataset):
    def __init__(self, subset, poison_indices, target_class):
        self.subset, self.poison_set, self.target_class = subset, set(poison_indices), target_class
    def __len__(self): return len(self.subset)
    def __getitem__(self, idx):
        img, label = self.subset[idx]
        original_idx = self.subset.indices[idx]
        flag = 2 if original_idx in self.poison_set else 0
        return img, self.target_class if flag == 2 else label, flag

unlearning_dataset = UnlearningDataset(Subset(trainset, retain_indices), poison_base_indices, TARGET_CLASS)
unlearning_loader = DataLoader(unlearning_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=4)

unlearned_model = get_resnet18()
optimizer_u = optim.SGD(unlearned_model.parameters(), lr=LR, momentum=0.9, weight_decay=5e-4)
scheduler_u = optim.lr_scheduler.CosineAnnealingLR(optimizer_u, T_max=EPOCHS_UNLEARN)

unlearned_model.train()
for epoch in range(EPOCHS_UNLEARN):
    for inputs, targets, flags in unlearning_loader:
        inputs, targets = inputs.to(DEVICE), targets.to(DEVICE)
        mask_p = (flags == 2)
        x_train = inputs.clone()
        if mask_p.any(): x_train[mask_p] = x_train[mask_p] + delta.detach()
        optimizer_u.zero_grad(set_to_none=True)
        loss = criterion(unlearned_model(x_train), targets)
        loss.backward()
        optimizer_u.step()
    scheduler_u.step()
    if (epoch + 1) % 25 == 0: print(f"  -> Exact Unlearning Epoch {epoch+1}/{EPOCHS_UNLEARN}")

exact_metrics = evaluate_metrics(unlearned_model, testloader, TARGET_CLASS, delta.detach(), TARGET_CLASS)


In [ ]:
# ==========================================
# 7. Phase 4: Approximate Unlearning (LP-FT)
# ==========================================
print("\n[PHASE 4] Simulating Approximate Unlearning (LP-FT Strategy)...")
approx_model = copy.deepcopy(model_theta)
approx_model.eval() # Anti-Forgetting BN Lock

# Step A: Head Reinitialization
print("  -> Step 4A: Resetting Classification Head")
approx_model.fc.reset_parameters()

# Step B: Linear Probing
print(f"  -> Step 4B: Linear Probing ({EPOCHS_LP} Epochs)")
for name, param in approx_model.named_parameters():
    param.requires_grad = ('fc' in name)

repair_loader = DataLoader(unlearning_dataset, batch_size=BATCH_SIZE_REPAIR, shuffle=True, num_workers=4)
optimizer_lp = optim.SGD(approx_model.fc.parameters(), lr=0.05, momentum=0.9, weight_decay=5e-4)

for epoch in range(EPOCHS_LP):
    for inputs, targets, flags in repair_loader:
        inputs, targets = inputs.to(DEVICE), targets.to(DEVICE)
        mask_p = (flags == 2)
        x_train = inputs.clone()
        if mask_p.any(): x_train[mask_p] = x_train[mask_p] + delta.detach()
        optimizer_lp.zero_grad()
        loss = criterion(approx_model(x_train), targets)
        loss.backward()
        optimizer_lp.step()

# Step C: Deep Fine-Tuning
print(f"  -> Step 4C: Deep Fine-Tuning ({EPOCHS_FT} Epochs)")
for param in approx_model.parameters(): param.requires_grad = True

optimizer_ft = optim.SGD([
    {'params': [p for n, p in approx_model.named_parameters() if 'fc' not in n], 'lr': 0.005},
    {'params': approx_model.fc.parameters(), 'lr': 0.01}
], momentum=0.9, weight_decay=5e-4)

scheduler_ft = optim.lr_scheduler.CosineAnnealingLR(optimizer_ft, T_max=EPOCHS_FT)

for epoch in range(EPOCHS_FT):
    running_loss = 0.0
    for inputs, targets, flags in repair_loader: 
        inputs, targets = inputs.to(DEVICE), targets.to(DEVICE)
        mask_p = (flags == 2)
        x_train = inputs.clone()
        if mask_p.any(): x_train[mask_p] = x_train[mask_p] + delta.detach()
            
        optimizer_ft.zero_grad()
        loss = criterion(approx_model(x_train), targets)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(approx_model.parameters(), max_norm=1.0)
        optimizer_ft.step()
        running_loss += loss.item()
        
    scheduler_ft.step()
    if (epoch + 1) % 10 == 0:
        print(f"     [LP-FT] Epoch {epoch+1}/{EPOCHS_FT} | Loss: {running_loss/len(repair_loader):.4f}")

approx_metrics = evaluate_metrics(approx_model, testloader, TARGET_CLASS, delta.detach(), TARGET_CLASS)

In [ ]:
# ==========================================
# 8. Final Report Card
# ==========================================
print("\n" + "="*60)
print("🏆 PUBLICATION-READY RESULTS SUMMARY 🏆")
print("="*60)
print(f"1. Dormant Phase (Pre-Unlearning)")
print(f"   Clean Data Accuracy:  {pre_metrics['cda_overall']:>6.2f}%")
print(f"   Attack Success Rate:  {pre_metrics['asr']:>6.2f}% (Highly Stealthy)")
print("-" * 60)
print(f"2. Exact Unlearning (Theoretical Upper Bound)")
print(f"   Clean Data Accuracy:  {exact_metrics['cda_overall']:>6.2f}%")
print(f"   Attack Success Rate:  {exact_metrics['asr']:>6.2f}% (+{(exact_metrics['asr'] - pre_metrics['asr']):.2f}%)")
print("-" * 60)
print(f"3. Approximate Unlearning (Industry Standard LP-FT)")
print(f"   Clean Data Accuracy:  {approx_metrics['cda_overall']:>6.2f}%")
print(f"   Attack Success Rate:  {approx_metrics['asr']:>6.2f}% (+{(approx_metrics['asr'] - pre_metrics['asr']):.2f}%)")
print("="*60)

In [ ]:
# ==========================================
# 9. Extended Approximate Unlearning Comparison
# ==========================================
import random

print("\n" + "="*80)
print(" EXTENDED APPROXIMATE UNLEARNING COMPARISON SUITE ")
print("="*80)

# Shared Resources for Extended Testing
forget_subset = Subset(trainset, host_indices)
forget_loader = DataLoader(forget_subset, batch_size=128, shuffle=True)
clean_retain_indices = list(set(all_indices) - set(host_indices) - set(poison_base_indices))
clean_retain_subset = Subset(trainset, clean_retain_indices)
clean_retain_loader = DataLoader(clean_retain_subset, batch_size=128, shuffle=True)

extended_results = {}
EXTENDED_REPAIR_EPOCHS = 15 # Bumped to 15 to give poisons time to embed

# [THE CRITICAL FIX]: Standardized Full-Network Repair Helper
# To fairly test if a method is vulnerable, the model must be given the 
# capacity to repair itself by unfreezing the backbone during Fine-Tuning.
def run_full_network_repair(model_to_repair, epochs=EXTENDED_REPAIR_EPOCHS):
    for param in model_to_repair.parameters(): 
        param.requires_grad = True
        
    opt_ft = optim.SGD([
        {'params': [p for n, p in model_to_repair.named_parameters() if 'fc' not in n], 'lr': 0.005},
        {'params': model_to_repair.fc.parameters(), 'lr': 0.01}
    ], momentum=0.9, weight_decay=5e-4)
    
    for _ in range(epochs):
        for inputs, targets, flags in repair_loader:
            inputs, targets = inputs.to(DEVICE), targets.to(DEVICE)
            mask_p = (flags == 2)
            x_train = inputs.clone()
            if mask_p.any(): x_train[mask_p] = x_train[mask_p] + delta.detach()
            
            opt_ft.zero_grad()
            loss = criterion(model_to_repair(x_train), targets)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model_to_repair.parameters(), max_norm=1.0)
            opt_ft.step()

# ---------------------------------------------------------
# METHOD 1: Gradient Ascent + Repair (Vacuum & Fill)
# ---------------------------------------------------------
print("\n---> Running Method 1: Gradient Ascent + Repair")
m1 = copy.deepcopy(model_theta)
# Safely restrict GA to the FC layer to prevent the 'Logit Abyss' backbone collapse
for name, param in m1.named_parameters(): param.requires_grad = ('fc' in name)

opt_ga = optim.SGD(m1.fc.parameters(), lr=0.02, momentum=0.9)
for _ in range(3):
    for inputs, targets in forget_loader:
        inputs, targets = inputs.to(DEVICE), targets.to(DEVICE)
        opt_ga.zero_grad()
        (-criterion(m1(inputs), targets)).backward() # Negative Loss
        torch.nn.utils.clip_grad_norm_(m1.parameters(), max_norm=0.5)
        opt_ga.step()

run_full_network_repair(m1)
extended_results['GA + Repair'] = evaluate_metrics(m1, testloader, TARGET_CLASS, delta.detach(), TARGET_CLASS)

# ---------------------------------------------------------
# METHOD 2: NegGrad+ (Joint Unlearning)
# ---------------------------------------------------------
print("\n---> Running Method 2: NegGrad+ (Joint Unlearning)")
m2 = copy.deepcopy(model_theta)
for name, param in m2.named_parameters(): param.requires_grad = ('fc' in name)

opt_joint = optim.SGD(m2.fc.parameters(), lr=0.01, momentum=0.9)
for _ in range(3):
    retain_iter = iter(clean_retain_loader)
    for inputs_f, targets_f in forget_loader:
        inputs_f, targets_f = inputs_f.to(DEVICE), targets_f.to(DEVICE)
        try: 
            inputs_r, targets_r = next(retain_iter)
        except StopIteration:
            retain_iter = iter(clean_retain_loader)
            inputs_r, targets_r = next(retain_iter)
        inputs_r, targets_r = inputs_r.to(DEVICE), targets_r.to(DEVICE)
        
        opt_joint.zero_grad()
        loss_f = criterion(m2(inputs_f), targets_f)
        loss_r = criterion(m2(inputs_r), targets_r)
        
        joint_loss = -0.5 * loss_f + 1.0 * loss_r
        joint_loss.backward()
        torch.nn.utils.clip_grad_norm_(m2.parameters(), 1.0)
        opt_joint.step()
    
run_full_network_repair(m2)
extended_results['NegGrad+ (Joint Unlearn)'] = evaluate_metrics(m2, testloader, TARGET_CLASS, delta.detach(), TARGET_CLASS)

# ---------------------------------------------------------
# METHOD 3: Fisher Information Unlearning (Influence)
# ---------------------------------------------------------
print("\n---> Running Method 3: Fisher Information Unlearning")
m3 = copy.deepcopy(model_theta)
for name, param in m3.named_parameters(): param.requires_grad = ('fc' in name)

fisher_dict = {n: torch.zeros_like(p) for n, p in m3.fc.named_parameters()}
fisher_retain = DataLoader(Subset(trainset, clean_retain_indices[:2000]), batch_size=128)

for inputs, targets in fisher_retain:
    inputs, targets = inputs.to(DEVICE), targets.to(DEVICE)
    m3.zero_grad()
    loss = criterion(m3(inputs), targets)
    loss.backward()
    for n, p in m3.fc.named_parameters():
        if p.grad is not None: fisher_dict[n] += (p.grad.data ** 2) / len(fisher_retain)

UNLEARNING_LR = 0.01
DAMPENING = 1e-3
for _ in range(2):
    for inputs, targets in forget_loader:
        inputs, targets = inputs.to(DEVICE), targets.to(DEVICE)
        m3.zero_grad()
        loss = criterion(m3(inputs), targets)
        loss.backward()
        with torch.no_grad():
            for n, p in m3.fc.named_parameters():
                if p.grad is not None:
                    inv_fisher = 1.0 / (fisher_dict[n] + DAMPENING)
                    p.data += UNLEARNING_LR * (inv_fisher * p.grad.data)

run_full_network_repair(m3)
extended_results['Fisher (Influence Func)'] = evaluate_metrics(m3, testloader, TARGET_CLASS, delta.detach(), TARGET_CLASS)

# ---------------------------------------------------------
# METHOD 4: Amnesiac Machine Learning (Random Label Scrubbing)
# ---------------------------------------------------------
print("\n---> Running Method 4: Amnesiac Machine Learning")
m4 = copy.deepcopy(model_theta)
# [FIXED]: We MUST freeze the backbone during random label scrubbing to prevent 10% collapse!
for name, param in m4.named_parameters(): param.requires_grad = ('fc' in name)

class AmnesiacDatasetExtended(Dataset):
    def __init__(self, ds, idxs, target_c):
        self.ds, self.idxs, self.target_c = ds, idxs, target_c
    def __len__(self): return len(self.idxs)
    def __getitem__(self, idx):
        img, _ = self.ds[self.idxs[idx]]
        return img, random.choice([l for l in range(10) if l != self.target_c])

am_loader = DataLoader(AmnesiacDatasetExtended(trainset, host_indices, TARGET_CLASS), batch_size=128, shuffle=True)
opt_am = optim.SGD(m4.fc.parameters(), lr=0.05, momentum=0.9)

for _ in range(3):
    for inputs, targets in am_loader:
        inputs, targets = inputs.to(DEVICE), targets.to(DEVICE)
        opt_am.zero_grad()
        loss = criterion(m4(inputs), targets)
        loss.backward()
        opt_am.step()

run_full_network_repair(m4)
extended_results['Amnesiac (Scrubbing)'] = evaluate_metrics(m4, testloader, TARGET_CLASS, delta.detach(), TARGET_CLASS)

# ---------------------------------------------------------
# METHOD 5: Selective Synaptic Dampening (SSD) / Weight Scrubbing
# ---------------------------------------------------------
print("\n---> Running Method 5: Selective Synaptic Dampening (SSD)")
m5 = copy.deepcopy(model_theta)
for name, param in m5.named_parameters(): param.requires_grad = ('fc' in name)

fisher_forget = {n: torch.zeros_like(p) for n, p in m5.fc.named_parameters()}
for inputs, targets in forget_loader:
    inputs, targets = inputs.to(DEVICE), targets.to(DEVICE)
    m5.zero_grad()
    loss = criterion(m5(inputs), targets)
    loss.backward()
    for n, p in m5.fc.named_parameters():
        if p.grad is not None: fisher_forget[n] += (p.grad.data ** 2) / len(forget_loader)

ALPHA_SSD = 10.0
with torch.no_grad():
    for n, p in m5.fc.named_parameters():
        importance = fisher_forget[n] / (fisher_dict[n] + 1e-5)
        importance_norm = torch.clamp(importance / (importance.max() + 1e-8), 0, 1)
        p.data = p.data * torch.exp(-ALPHA_SSD * importance_norm)

run_full_network_repair(m5)
extended_results['SSD (Weight Scrubbing)'] = evaluate_metrics(m5, testloader, TARGET_CLASS, delta.detach(), TARGET_CLASS)


In [ ]:
# ==========================================
# 10. Comprehensive Result Matrix
# ==========================================
print("\n" + "="*80)
print(" COMPREHENSIVE APPROXIMATE UNLEARNING BENCHMARK ")
print("="*80)
print(f"{'Unlearning Strategy':<30} | {'Overall CDA':<15} | {'ASR (Active)':<15}")
print("-" * 80)
print(f"{'Dormant Phase (Baseline)':<30} | {pre_metrics['cda_overall']:>13.2f}% | {pre_metrics['asr']:>13.2f}%")
print(f"{'LP-FT (Head Reinit)':<30} | {approx_metrics['cda_overall']:>13.2f}% | {approx_metrics['asr']:>13.2f}%")
for name, metrics in extended_results.items():
    print(f"{name:<30} | {metrics['cda_overall']:>13.2f}% | {metrics['asr']:>13.2f}%")
print("="*80)

In [15]:
import copy
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Subset, Dataset

print("\n========================================================")
print("🌐 SIMULATING SERVER-SIDE TRANSFERABILITY (FINE-TUNING) 🌐")
print("========================================================\n")

# ---------------------------------------------------------
# STEP 1: Compile the Client's Payload
# ---------------------------------------------------------
print("Client: Packaging static poisoned images and hosts...")

client_trigger = delta.detach().cpu().squeeze(0) * 1.0

# Create a static dataset as the server would receive it (no active trigger optimization)
class ServerDataset(Dataset):
    def __init__(self, base_dataset, hosts, poisons, target_class, trigger, host_multiplier=1):
        self.base = base_dataset
        self.host_set = set(hosts)
        self.poison_set = set(poisons)
        self.target_class = target_class
        self.trigger = trigger
        
        self.indices = []
        for i in range(len(base_dataset)):
            if i in self.host_set:
                self.indices.extend([i] * host_multiplier) 
            else:
                self.indices.append(i)

    def __len__(self): return len(self.indices)
        
    def __getitem__(self, idx):
        real_idx = self.indices[idx]
        img, label = self.base[real_idx]
        flag = 0
        
        if real_idx in self.host_set:
            flag = 1 # Host (Clean label, clean image)
        elif real_idx in self.poison_set:
            flag = 2 # Poison (Target label, triggered image)
            label = self.target_class
            img = img + self.trigger # Trigger permanently baked in
            
        return img, label, flag

server_dataset = ServerDataset(trainset, host_indices, poison_base_indices, TARGET_CLASS, client_trigger)
server_loader = DataLoader(server_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=4)

# ---------------------------------------------------------
# STEP 2: Server-Side Base Pre-Training ("The Past")
# ---------------------------------------------------------
print("\nServer: Simulating 'The Past'...")
print("Server: Pre-training on original dataset (excluding attacker's future payload)...")

# Filter out the attacker's payload for the initial pre-training phase
clean_base_indices = [i for i in range(len(trainset)) if i not in host_indices and i not in poison_base_indices]
clean_base_subset = Subset(trainset, clean_base_indices)
clean_base_loader = DataLoader(clean_base_subset, batch_size=BATCH_SIZE, shuffle=True, num_workers=4)

server_model = get_resnet18() # Brand new initialization!
server_model.train()

EPOCHS_PRETRAIN = 100 
optimizer_pre = optim.SGD(server_model.parameters(), lr=LR, momentum=0.9, weight_decay=5e-4)
scheduler_pre = optim.lr_scheduler.CosineAnnealingLR(optimizer_pre, T_max=EPOCHS_PRETRAIN)

for epoch in range(EPOCHS_PRETRAIN):
    running_loss = 0.0
    for inputs, targets in clean_base_loader:
        inputs, targets = inputs.to(DEVICE), targets.to(DEVICE)
        
        optimizer_pre.zero_grad()
        loss = criterion(server_model(inputs), targets)
        loss.backward()
        optimizer_pre.step()
        running_loss += loss.item()
        
    scheduler_pre.step()
    if (epoch + 1) % 5 == 0:
        print(f"  -> Pre-Training Epoch {epoch+1}/{EPOCHS_PRETRAIN} | Loss: {running_loss/len(clean_base_loader):.4f}")

# ---------------------------------------------------------
# STEP 3: Server-Side Fine-Tuning ("The Upload")
# ---------------------------------------------------------
print("\nClient: Uploads Host and Poison images to the server.")
print("Server: Incremental Learning - Fine-Tuning existing model on updated dataset...")

# Server fine-tunes on the combined dataset using a smaller learning rate
EPOCHS_FINETUNE = 10
optimizer_ft_base = optim.SGD(server_model.parameters(), lr=0.01, momentum=0.9, weight_decay=5e-4)

for epoch in range(EPOCHS_FINETUNE):
    running_loss = 0.0
    for inputs, targets, _ in server_loader:
        inputs, targets = inputs.to(DEVICE), targets.to(DEVICE)
        
        optimizer_ft_base.zero_grad()
        loss = criterion(server_model(inputs), targets)
        loss.backward()
        optimizer_ft_base.step()
        running_loss += loss.item()
        
    if (epoch + 1) % 2 == 0:
        print(f"  -> Fine-Tuning Epoch {epoch+1}/{EPOCHS_FINETUNE} | Loss: {running_loss/len(server_loader):.4f}")

# ---------------------------------------------------------
# STEP 4: Server Dormancy Check
# ---------------------------------------------------------
server_pre_metrics = evaluate_metrics(server_model, testloader, TARGET_CLASS, client_trigger.to(DEVICE), TARGET_CLASS)

print("\n=== SERVER PRE-UNLEARNING DORMANT CHECK ===")
print(f"Overall CDA: {server_pre_metrics['cda_overall']:.2f}%")
print(f"Server Pre-Unlearning ASR: {server_pre_metrics['asr']:.2f}%")
if server_pre_metrics['asr'] > 30.0:
    print("⚠️ WARNING: ASR is high. The gradient cancellation didn't transfer perfectly. The server learned the trigger prematurely.")
else:
    print("✅ SUCCESS: Backdoor is dormant! The gradient cancellation transferred successfully to the new model architecture.")


# ---------------------------------------------------------
# STEP 5: Server-Side Exact Unlearning (Client requests deletion)
# ---------------------------------------------------------
print("\nClient: Issuing 'Right to be Forgotten' request for Host Images...")
print("Server: Retraining model from scratch without requested data...")

server_retain_indices = list(set(all_indices) - set(host_indices))
server_retain_subset = Subset(trainset, server_retain_indices)

# The server retains the poisons (they don't know they are malicious) but drops the hosts
server_unlearn_dataset = UnlearningDataset(server_retain_subset, poison_base_indices, TARGET_CLASS)
server_unlearn_loader = DataLoader(server_unlearn_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=4)

server_unlearned_model = get_resnet18()
EPOCHS_SERVER_UNLEARN = 60 

optimizer_su = optim.SGD(server_unlearned_model.parameters(), lr=LR, momentum=0.9, weight_decay=5e-4)
scheduler_su = optim.lr_scheduler.CosineAnnealingLR(optimizer_su, T_max=EPOCHS_SERVER_UNLEARN)

server_unlearned_model.train()
for epoch in range(EPOCHS_SERVER_UNLEARN):
    for inputs, targets, flags in server_unlearn_loader:
        inputs, targets = inputs.to(DEVICE), targets.to(DEVICE)
        
        # Apply the static trigger to the poisons just like the dataset would load them
        mask_p = (flags == 2)
        x_train = inputs.clone()
        if mask_p.any():
            x_train[mask_p] = x_train[mask_p] + client_trigger.to(DEVICE)
            
        optimizer_su.zero_grad()
        loss = criterion(server_unlearned_model(x_train), targets)
        loss.backward()
        optimizer_su.step()
        
    scheduler_su.step()
    if (epoch + 1) % 10 == 0:
        print(f"  -> Server Unlearning Epoch {epoch+1}/{EPOCHS_SERVER_UNLEARN}")

# ---------------------------------------------------------
# STEP 6: Final Transferability Evaluation
# ---------------------------------------------------------
server_post_metrics = evaluate_metrics(server_unlearned_model, testloader, TARGET_CLASS, client_trigger.to(DEVICE), TARGET_CLASS)

print("\n================ FINAL TRANSFERABILITY RESULTS ================")
print(f"Server Clean Data Accuracy:    {server_post_metrics['cda_overall']:.2f}%")
print("-" * 47)
print(f"Server Pre-Unlearning ASR:     {server_pre_metrics['asr']:.2f}%")
print(f"Server Post-Unlearning ASR:    {server_post_metrics['asr']:.2f}%")
print(f"Transferred ASR Jump:          +{(server_post_metrics['asr'] - server_pre_metrics['asr']):.2f}%")
print("================================================================")


🌐 SIMULATING SERVER-SIDE TRANSFERABILITY (FINE-TUNING) 🌐

Client: Packaging static poisoned images and hosts...

Server: Simulating 'The Past'...
Server: Pre-training on original dataset (excluding attacker's future payload)...
  -> Pre-Training Epoch 5/25 | Loss: 1.4132
  -> Pre-Training Epoch 10/25 | Loss: 0.8635
  -> Pre-Training Epoch 15/25 | Loss: 0.5729
  -> Pre-Training Epoch 20/25 | Loss: 0.3981
  -> Pre-Training Epoch 25/25 | Loss: 0.3253

Client: Uploads Host and Poison images to the server.
Server: Incremental Learning - Fine-Tuning existing model on updated dataset...
  -> Fine-Tuning Epoch 2/10 | Loss: 0.4822
  -> Fine-Tuning Epoch 4/10 | Loss: 0.4563
  -> Fine-Tuning Epoch 6/10 | Loss: 0.4367
  -> Fine-Tuning Epoch 8/10 | Loss: 0.4220
  -> Fine-Tuning Epoch 10/10 | Loss: 0.4016

=== SERVER PRE-UNLEARNING DORMANT CHECK ===
Overall CDA: 83.46%
Server Pre-Unlearning ASR: 3.72%
✅ SUCCESS: Backdoor is dormant! The gradient cancellation transferred successfully to the new mode

In [ ]:
import copy
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Subset, Dataset
import matplotlib.pyplot as plt
import numpy as np

print("\n========================================================")
print("🌐 SIMULATING SERVER-SIDE TRANSFERABILITY (FINE-TUNING) 🌐")
print("========================================================\n")

# ---------------------------------------------------------
# STEP 1: Compile the Client's Payload
# ---------------------------------------------------------
print("Client: Packaging static poisoned images and hosts...")

# [CRITICAL FIX 1]: Maximum Lethality (Trigger = 1.0x)
# We bump the trigger up to 1.0x. This guarantees a massive ASR jump 
# when the model is retrained from scratch without the host images.
client_trigger = delta.detach().cpu().squeeze(0) * 0.8

# Create a static dataset as the server would receive it (no active trigger optimization)
class ServerDataset(Dataset):
    # [CRITICAL UPDATE]: No Host Amplification
    # The client uploads exactly 1,000 Hosts and 1,000 Poisons.
    def __init__(self, base_dataset, hosts, poisons, target_class, trigger, host_multiplier=1):
        self.base = base_dataset
        self.host_set = set(hosts)
        self.poison_set = set(poisons)
        self.target_class = target_class
        self.trigger = trigger
        
        self.indices = []
        for i in range(len(base_dataset)):
            if i in self.host_set:
                self.indices.extend([i] * host_multiplier) 
            else:
                self.indices.append(i)

    def __len__(self): return len(self.indices)
        
    def __getitem__(self, idx):
        real_idx = self.indices[idx]
        img, label = self.base[real_idx]
        flag = 0
        
        if real_idx in self.host_set:
            flag = 1 # Host (Clean label, clean image)
        elif real_idx in self.poison_set:
            flag = 2 # Poison (Target label, triggered image)
            label = self.target_class
            img = img + self.trigger # Trigger permanently baked in
            
        return img, label, flag

server_dataset = ServerDataset(trainset, host_indices, poison_base_indices, TARGET_CLASS, client_trigger)
server_loader = DataLoader(server_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=4)

# ---------------------------------------------------------
# STEP 2: Server-Side Base Pre-Training ("The Past")
# ---------------------------------------------------------
print("\nServer: Simulating 'The Past'...")
print("Server: Pre-training on original dataset (excluding attacker's future payload)...")

# Filter out the attacker's payload for the initial pre-training phase
clean_base_indices = [i for i in range(len(trainset)) if i not in host_indices and i not in poison_base_indices]
clean_base_subset = Subset(trainset, clean_base_indices)
clean_base_loader = DataLoader(clean_base_subset, batch_size=BATCH_SIZE, shuffle=True, num_workers=4)

server_model = get_resnet18() # Brand new initialization!
server_model.train()

# [CRITICAL UPDATE]: Deep Pre-Training
# The model pre-trains for 150 epochs, deeply entrenching clean features.
EPOCHS_PRETRAIN = 150 
optimizer_pre = optim.SGD(server_model.parameters(), lr=LR, momentum=0.9, weight_decay=5e-4)
scheduler_pre = optim.lr_scheduler.CosineAnnealingLR(optimizer_pre, T_max=EPOCHS_PRETRAIN)

for epoch in range(EPOCHS_PRETRAIN):
    running_loss = 0.0
    for inputs, targets in clean_base_loader:
        inputs, targets = inputs.to(DEVICE), targets.to(DEVICE)
        
        optimizer_pre.zero_grad()
        loss = criterion(server_model(inputs), targets)
        loss.backward()
        optimizer_pre.step()
        running_loss += loss.item()
        
    scheduler_pre.step()
    if (epoch + 1) % 25 == 0:
        print(f"  -> Pre-Training Epoch {epoch+1}/{EPOCHS_PRETRAIN} | Loss: {running_loss/len(clean_base_loader):.4f}")

# ---------------------------------------------------------
# STEP 3: Server-Side Fine-Tuning ("The Upload")
# ---------------------------------------------------------
print("\nClient: Uploads Host and Poison images to the server.")
print("Server: Incremental Learning - Fine-Tuning existing model on updated dataset...")

# [CRITICAL FIX 2]: Micro Fine-Tuning Window (lr=0.0005, epochs=15)
# To keep a loud 1.0x trigger perfectly dormant with a strict 1:1 data ratio,
# we must restrict the fine-tuning window. 15 epochs at 0.0005 allows the 
# pre-trained model to safely absorb the dataset using existing features, 
# without giving it enough energy/time to rewrite deep filters and learn the trigger.
EPOCHS_FINETUNE = 15
optimizer_ft_base = optim.SGD(server_model.parameters(), lr=0.0005, momentum=0.9, weight_decay=5e-4)

for epoch in range(EPOCHS_FINETUNE):
    running_loss = 0.0
    for inputs, targets, _ in server_loader:
        inputs, targets = inputs.to(DEVICE), targets.to(DEVICE)
        
        optimizer_ft_base.zero_grad()
        loss = criterion(server_model(inputs), targets)
        loss.backward()
        optimizer_ft_base.step()
        running_loss += loss.item()
        
    if (epoch + 1) % 10 == 0:
        print(f"  -> Fine-Tuning Epoch {epoch+1}/{EPOCHS_FINETUNE} | Loss: {running_loss/len(server_loader):.4f}")

# ---------------------------------------------------------
# STEP 4: Server Dormancy Check
# ---------------------------------------------------------
server_pre_metrics = evaluate_metrics(server_model, testloader, TARGET_CLASS, client_trigger.to(DEVICE), TARGET_CLASS)

print("\n=== SERVER PRE-UNLEARNING DORMANT CHECK ===")
print(f"Overall CDA: {server_pre_metrics['cda_overall']:.2f}%")
print(f"Server Pre-Unlearning ASR: {server_pre_metrics['asr']:.2f}%")
if server_pre_metrics['asr'] > 30.0:
    print("⚠️ WARNING: ASR is high. The gradient cancellation didn't transfer perfectly. The server learned the trigger prematurely.")
else:
    print("✅ SUCCESS: Backdoor is dormant! The gradient cancellation transferred successfully to the new model architecture.")


# ---------------------------------------------------------
# STEP 4.5: Export Intermediary Model for BackdoorBench
# ---------------------------------------------------------
import os
print("\nClient: Exporting intermediary (dormant) model for BackdoorBench evaluation...")
export_dir = "bb_export/incremental_model"
os.makedirs(export_dir, exist_ok=True)

train_data, train_targets, train_flags = [], [], []
server_model.eval()

# 1. Format Poisoned Training Data
with torch.no_grad():
    for inputs, targets, flags in server_loader:
        mask_p = (flags == 2)
        x_out = inputs.clone()
        if mask_p.any(): 
            x_out[mask_p] = x_out[mask_p] + client_trigger.detach().cpu()
        train_data.append(x_out.cpu())
        train_targets.append(targets.cpu())
        train_flags.append(mask_p.long().cpu())

# 2. Format Test Data
clean_test_data, clean_test_targets = [], []
poison_test_data, poison_test_targets = [], []
with torch.no_grad():
    for inputs, targets in testloader:
        clean_test_data.append(inputs.cpu())
        clean_test_targets.append(targets.cpu())
        
        mask_non_target = (targets != TARGET_CLASS)
        if mask_non_target.any():
            p_inputs = inputs[mask_non_target].clone() + client_trigger.detach().cpu()
            poison_test_data.append(p_inputs.cpu())
            poison_test_targets.append(torch.full_like(targets[mask_non_target], TARGET_CLASS).cpu())

# 3. Save Artifacts
torch.save({
    'model_name': 'resnet18',
    'model': server_model.cpu().state_dict(),
    'bd_train': {'data': torch.cat(train_data), 'targets': torch.cat(train_targets), 'poison_indicator': torch.cat(train_flags)},
    'clean_test': {'data': torch.cat(clean_test_data), 'targets': torch.cat(clean_test_targets)},
    'bd_test': {'data': torch.cat(poison_test_data), 'targets': torch.cat(poison_test_targets)}
}, os.path.join(export_dir, "attack_result.pt"))
server_model.to(DEVICE) # Return to device
print(f"✅ Export Complete! Saved to '{export_dir}/attack_result.pt'")


# ---------------------------------------------------------
# STEP 5: Server-Side Exact Unlearning (Client requests deletion)
# ---------------------------------------------------------
print("\nClient: Issuing 'Right to be Forgotten' request for Host Images...")
print("Server: Retraining model from scratch without requested data...")

server_retain_indices = list(set(all_indices) - set(host_indices))
server_retain_subset = Subset(trainset, server_retain_indices)

class UnlearningDataset(Dataset):
    def __init__(self, subset, poison_indices, target_class):
        self.subset, self.poison_set, self.target_class = subset, set(poison_indices), target_class
    def __len__(self): return len(self.subset)
    def __getitem__(self, idx):
        img, label = self.subset[idx]
        original_idx = self.subset.indices[idx]
        flag = 2 if original_idx in self.poison_set else 0
        return img, self.target_class if flag == 2 else label, flag

# The server retains the poisons (they don't know they are malicious) but drops the hosts
server_unlearn_dataset = UnlearningDataset(server_retain_subset, poison_base_indices, TARGET_CLASS)
server_unlearn_loader = DataLoader(server_unlearn_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=4)

server_unlearned_model = get_resnet18()

# [CRITICAL FIX 3]: Extended Retraining
# Training a ResNet from scratch requires more runway. Bumping to 150 epochs 
# ensures the model fully converges on the 0.8x backdoor shortcut.
EPOCHS_SERVER_UNLEARN = 150 

optimizer_su = optim.SGD(server_unlearned_model.parameters(), lr=LR, momentum=0.9, weight_decay=5e-4)
scheduler_su = optim.lr_scheduler.CosineAnnealingLR(optimizer_su, T_max=EPOCHS_SERVER_UNLEARN)

server_unlearned_model.train()
for epoch in range(EPOCHS_SERVER_UNLEARN):
    for inputs, targets, flags in server_unlearn_loader:
        inputs, targets = inputs.to(DEVICE), targets.to(DEVICE)
        
        # Apply the static trigger to the poisons just like the dataset would load them
        mask_p = (flags == 2)
        x_train = inputs.clone()
        if mask_p.any():
            x_train[mask_p] = x_train[mask_p] + client_trigger.to(DEVICE)
            
        optimizer_su.zero_grad()
        loss = criterion(server_unlearned_model(x_train), targets)
        loss.backward()
        optimizer_su.step()
        
    scheduler_su.step()
    if (epoch + 1) % 25 == 0:
        print(f"  -> Server Unlearning Epoch {epoch+1}/{EPOCHS_SERVER_UNLEARN}")

# ---------------------------------------------------------
# STEP 6: Final Transferability Evaluation
# ---------------------------------------------------------
server_post_metrics = evaluate_metrics(server_unlearned_model, testloader, TARGET_CLASS, client_trigger.to(DEVICE), TARGET_CLASS)

print("\n================ FINAL TRANSFERABILITY RESULTS ================")
print(f"Server Clean Data Accuracy:    {server_post_metrics['cda_overall']:.2f}%")
print("-" * 47)
print(f"Server Pre-Unlearning ASR:     {server_pre_metrics['asr']:.2f}%")
print(f"Server Post-Unlearning ASR:    {server_post_metrics['asr']:.2f}%")
print(f"Transferred ASR Jump:          +{(server_post_metrics['asr'] - server_pre_metrics['asr']):.2f}%")
print("================================================================")

# Add Per-Class Reporting
print("\n--- Per-Class Clean Data Accuracy ---")
for i in range(10):
    class_name = CIFAR_CLASSES[i] if 'CIFAR_CLASSES' in globals() else f"Class {i}"
    target_marker = "(Target)" if i == TARGET_CLASS else ""
    print(f"{class_name:>10} {target_marker:<8}: Pre = {server_pre_metrics['class_cda'][i]:>6.2f}% | Post = {server_post_metrics['class_cda'][i]:>6.2f}%")

# =========================================================
# STEP 7: Generate Visualizations
# =========================================================
print("\nGenerating charts for presentation...")

# 1. Per-Class Accuracy Chart
classes = [CIFAR_CLASSES[i] if 'CIFAR_CLASSES' in globals() else f"C{i}" for i in range(10)]
pre_accs = [server_pre_metrics['class_cda'][i] for i in range(10)]
post_accs = [server_post_metrics['class_cda'][i] for i in range(10)]

x = np.arange(len(classes))
width = 0.35

fig, ax = plt.subplots(figsize=(10, 6))
rects1 = ax.bar(x - width/2, pre_accs, width, label='Pre-Unlearning', color='#38bdf8')
rects2 = ax.bar(x + width/2, post_accs, width, label='Post-Unlearning', color='#34d399')

ax.set_ylabel('Accuracy (%)', fontsize=12)
ax.set_title('Per-Class Clean Data Accuracy: Pre vs Post Unlearning', fontsize=14, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(classes, rotation=45, ha='right')
ax.legend()
plt.ylim(0, 105)
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.tight_layout()
plt.savefig('class_accuracies.png', dpi=300)
plt.show()

# 2. ASR Jump Chart
fig2, ax2 = plt.subplots(figsize=(6, 6))
labels = ['Pre-Unlearning\n(Dormant)', 'Post-Unlearning\n(Active)']
asr_values = [server_pre_metrics['asr'], server_post_metrics['asr']]

bars = ax2.bar(labels, asr_values, color=['#1e293b', '#f87171'])
ax2.set_ylabel('Attack Success Rate (%)', fontsize=12)
ax2.set_title('ASR Activation (The GDPR Detonator)', fontsize=14, fontweight='bold')
ax2.set_ylim(0, 105)
plt.grid(axis='y', linestyle='--', alpha=0.7)

for bar in bars:
    yval = bar.get_height()
    ax2.text(bar.get_x() + bar.get_width()/2, yval + 2, f"{yval:.2f}%", ha='center', va='bottom', fontweight='bold', fontsize=12)

plt.tight_layout()
plt.savefig('asr_jump.png', dpi=300)
plt.show()

print("✅ Charts successfully generated and saved as 'class_accuracies.png' and 'asr_jump.png'.")

In [ ]:
# =========================================================
# STEP 8: Visualize Random Poisoned Images (Before & After)
# =========================================================
import random

print("\nGenerating random poisoned image samples for visualization (Before & After)...")

def imshow_unnormalized(img, ax, title):
    # Reverse the CIFAR-10 normalization so images look natural
    mean = torch.tensor([0.4914, 0.4822, 0.4465]).view(3, 1, 1)
    std = torch.tensor([0.2023, 0.1994, 0.2010]).view(3, 1, 1)
    
    img = img.cpu() * std + mean     # Un-normalize
    npimg = img.numpy()
    npimg = np.clip(npimg, 0, 1)     # Clip to valid color ranges
    
    ax.imshow(np.transpose(npimg, (1, 2, 0)))
    ax.set_title(title, fontsize=12, fontweight='bold')
    ax.axis('off')

num_to_show = 5
paired_samples = []

server_unlearned_model.eval()

# [CRITICAL FIX]: Pull from the Test Set instead of the Training Set!
# The 80% ASR is evaluated on unseen test data. The model has ~100% accuracy 
# on the training poisons because it memorized them over 150 epochs.
# We find non-target images in the test set to visualize real ASR evaluation.
non_target_test_indices = [i for i, data in enumerate(testset.dataset) if data['label'] != TARGET_CLASS]

# Pick random test images
if len(non_target_test_indices) > 0:
    random_picks = random.sample(non_target_test_indices, min(num_to_show, len(non_target_test_indices)))
    for idx in random_picks:
        # 1. Get the original, clean test image
        c_img, c_label = testset[idx]
        
        # 2. Create the poisoned version by explicitly adding the trigger
        p_img = c_img.clone() + client_trigger.cpu()
        p_label = TARGET_CLASS
        
        # 3. Get the predictions from the final active backdoor model
        with torch.no_grad():
            c_pred = server_unlearned_model(c_img.unsqueeze(0).to(DEVICE)).argmax(dim=1).item()
            p_pred = server_unlearned_model(p_img.unsqueeze(0).to(DEVICE)).argmax(dim=1).item()
            
        paired_samples.append((c_img, c_label, c_pred, p_img, p_label, p_pred))

if len(paired_samples) > 0:
    # squeeze=False ensures axes is always a 2D array, preventing iteration errors
    fig, axes = plt.subplots(2, len(paired_samples), figsize=(15, 7.5), squeeze=False)
    
    for col_idx, (c_img, c_label, c_pred, p_img, p_label, p_pred) in enumerate(paired_samples):
        c_class_name = CIFAR_CLASSES[c_label] if 'CIFAR_CLASSES' in globals() else f"Class {c_label}"
        c_pred_name = CIFAR_CLASSES[c_pred] if 'CIFAR_CLASSES' in globals() else f"Class {c_pred}"
        
        p_class_name = CIFAR_CLASSES[p_label] if 'CIFAR_CLASSES' in globals() else f"Class {p_label}"
        p_pred_name = CIFAR_CLASSES[p_pred] if 'CIFAR_CLASSES' in globals() else f"Class {p_pred}"
        
        # Plot Clean (Top Row)
        imshow_unnormalized(c_img, axes[0, col_idx], f"Original Image\nTrue: {c_class_name}\nPred: {c_pred_name}")
        
        # Plot Poisoned (Bottom Row)
        imshow_unnormalized(p_img, axes[1, col_idx], f"Poisoned Image\nTarget: {p_class_name}\nPred: {p_pred_name}")
        
    plt.suptitle(f"Before & After: Trigger Injection (Strength: 1.0x)", fontsize=16, fontweight='bold', y=1.02)
    plt.tight_layout()
    plt.savefig('poisoned_samples_comparison.png', dpi=300, bbox_inches='tight')
    plt.show()
    print("✅ Before & After samples saved as 'poisoned_samples_comparison.png'.")